# Pareja B · Idea 5 — Pistas de texto (zero-shot)

Frases por género ("a poster of a mystery film"…); CLIP mide qué tanto se parece el cartel a cada una (similitud coseno) y esos 17 números entran a la cabeza junto al vector. CLIP no genera nada: solo compara. Solo sirve con codificadores con texto (CLIP/SigLIP, no DINOv2).

**Dónde se programa:** `zeroshot.py` (`FRASES`, `similitudes`, `calcular`) y la variante `zeroshot` en `codificadores.py` (para que el predictor también la calcule). La lógica va en los `.py` (se revisa en el Pull Request); este
notebook solo la corre. **Listo cuando** el experimento imagen + pistas está en validación.

**Qué esperar:** cada pista es una función *lineal* del vector de imagen, así que con una cabeza lineal las pistas no agregan información nueva, solo un mejor punto de partida. Por eso se corre también `B5_zeroshot_mlp`, donde sí pueden combinarse con la imagen. Si ninguna mejora, es un resultado válido: se registra y se explica.

Antes, una vez para todo el equipo: `00_reproducir_base` (la base `v0.1.0`).


## Preparar el entorno

**GPU (T4)**: *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. La cuota gratis se acaba: calculen primero lo prioritario y repartan el cómputo entre sus cuentas.

- `P2M_DATOS`: dataset compartido (solo lectura). `P2M_SALIDAS`: carpeta **compartida del equipo**
  (`poster2movie_clasificador`), donde quedan vectores, modelos y predicciones para las otras parejas.
  Agreguen las dos a *Mi unidad* como acceso directo.
- Trabaja en la rama **`B-zeroshot`** (se crea desde `main` si no existe).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"            # solo lectura
os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_clasificador"   # compartida del equipo
REPO = "11909/Poster2Movie-Clasificador"
RAMA = "B-zeroshot"
DESTINO = "/content/clasificador"
if not os.path.isdir(DESTINO):
    !git clone -q --recurse-submodules https://github.com/{REPO}.git {DESTINO}
%cd {DESTINO}
!git fetch -q origin
!git checkout -q {RAMA} 2>/dev/null || git checkout -q -b {RAMA} origin/main
!git pull -q origin {RAMA} 2>/dev/null; git submodule update -q --init
!pip install -q -r requirements.txt
!nvidia-smi -L

In [ ]:
# ¿pasan las pruebas de la idea? (incluye que el predictor dé lo mismo que `calcular`)
!python -m pytest -q tests/test_pareja_b.py -k "similitudes or frases or zeroshot or predictor or b5 or diagnostico"


## 1. Pistas (GPU solo para codificar las frases; los carteles ya tienen vector)

In [ ]:
!python -m clasificador.zeroshot clip_b32

## 2. ¿Qué tan informativa es cada pista? (sin entrenar)

AUC de cada pista contra su género en **validación**: 0.5 no distingue nada, 1.0 separa perfecto. Si un género débil sale cerca de 0.5, hay que reescribir sus frases en `FRASES` (`zeroshot.py`) y volver a correr la celda 1. Se afinan frases mirando validación, nunca test.


In [ ]:
!python -m clasificador.zeroshot clip_b32 --diagnostico


## 3. Experimentos imagen + pistas (cabeza lineal y MLP)


In [ ]:
# entrenar en train, evaluar en VALIDACIÓN y comparar contra base (los 4 débiles salen primero)
for nombre in ["B5_zeroshot", "B5_zeroshot_mlp"]:
    !python -m clasificador.entrenar experimentos/{nombre}.yaml
    !python -m clasificador.evaluar {nombre}
    !python -m clasificador.comparar base {nombre}
!python -m clasificador.ranking B5_


## Subir a la rama `B-zeroshot`

En Colab el repo se borra al cerrar: suban `resultados/` antes. Se sube a **`B-zeroshot`**, nunca a
`main`; luego abran el Pull Request (lo revisa alguien de **otra** pareja). Necesitan un token de
GitHub con permiso de escritura (*Settings → Developer settings → Personal access tokens → Tokens (classic)*, con el permiso `repo` (los *fine-grained* no sirven: el repo es de otra cuenta personal)).

In [ ]:
from getpass import getpass
!git config user.name "{input('Tu nombre: ')}"
!git config user.email "{input('Tu correo de GitHub: ')}"
!git add experimentos resultados clasificador tests
!git commit -q -m "{RAMA}: {input('Qué hiciste (mensaje del commit): ')}"
token = getpass("Token de GitHub: ")
!git push -q https://{token}@github.com/{REPO}.git HEAD:{RAMA}
del token
print(f"Abre el Pull Request: https://github.com/{REPO}/compare/main...{RAMA}")